In [3]:
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType, Collection, utility
)
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
import spacy

In [42]:
connections.connect("default", host="localhost", port="19530")

fields = [
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535, is_primary=True),
    FieldSchema(name="year", dtype=DataType.INT64),
    FieldSchema(name="month", dtype=DataType.INT64),
    FieldSchema(name="day", dtype=DataType.INT64),
    FieldSchema(
        name="words",
        dtype=DataType.ARRAY,
        element_type=DataType.VARCHAR,
        max_length=64,
        max_capacity=2000
    ),
    FieldSchema(
        name="entities",
        dtype=DataType.ARRAY,
        element_type=DataType.VARCHAR,
        max_length=64,
        max_capacity=100
    ),
]


schema = CollectionSchema(fields=fields, description="Collection of Russian Speeches")

collection_name = "russian_speeches"
if collection_name in utility.list_collections():
    utility.drop_collection(collection_name)

collection = Collection(name=collection_name, schema=schema)
collection.create_index(
    field_name="embedding",
    index_params={
        "index_type": "IVF_FLAT",
        "metric_type": "COSINE",
        "params": {"nlist": 1024}
    }
)
print(f"Collection `{collection_name}` created successfully.")

Collection `russian_speeches` created successfully.


In [ ]:
collection.load()

results = collection.query(
    expr="year == 2024",
    output_fields=["year", "month", "day", "embedding", "entities"]
)

In [ ]:
data = pd.read_json("data/putin_complete.json")
stop_words = set(stopwords.words('english'))
punctuation = [".", ",", "?", "!", ":", "`", "'", "(", ")", "[", "]", "/", '’', "-", "’s", "\"", ";", "i", " ", "–", "%", "*", "...", "…"]
lemmatize = spacy.load("en_core_web_sm")

In [ ]:
for id, speech in data.iterrows():
    print(speech["date"], speech["transcript_filtered"])

1999-12-31 00:01:00 Dear friends, On New Year’s Eve, my family and I planned to gather round the TV, just as you probably did, to listen to the address by President Boris Yeltsin. But things took a different turn. On December 31, 1999, Russia’s first president decided to resign. He has asked me to address the Russian people today. The powers of the head of state have been turned over to me today. The presidential election will be held in three months. I assure you that there will be no vacuum of power, not for a minute. I promise you that any attempts to act contrary to the Russian law and constitution will be cut short. The state will stand firm to protect the freedom of speech, the freedom of conscience, the freedom of the mass media, ownership rights, these fundamental elements of a civilised society. The Armed Forces, the Federal Frontier Service, and law-enforcement agencies are working in the usual regime. The state continues to uphold the safety of every Russian citizen. When ma